# 03 · Data Preparation : préparation des 3 jeux de données
**Projet : E-Commerce Intelligence Platform** · Dataset Olist Brazilian E-Commerce · CRISP-DM, phase 3

Ce notebook transforme les 9 tables brutes en **3 jeux de données prêts pour la modélisation** :

| Fichier produit | Modèle | Granularité | Cible |
|---|---|---|---|
| `model1_delivery_duration.csv` | 1. Durée de livraison (régression) | 1 ligne par commande livrée | `delivery_days` |
| `model2_late_risk.csv` | 2. Risque de retard (classification) | 1 ligne par commande livrée | `is_late` |
| `model3_customer_segmentation.csv` | 3. Segmentation des clients (clustering) | 1 ligne par client réel | aucune |

**Principes appliqués**
- Les corrections décidées en Data Understanding sont appliquées ici (et seulement ici).
- **Aucune fuite de données** : les variables des modèles 1 et 2 sont toutes connues à l'achat. Les statistiques de vendeur et de trajet sont calculées uniquement sur des livraisons **déjà terminées** avant la date d'achat.
- **Découpage temporel** (train / valid / test), sans mélange aléatoire.
- Les paramètres (période, dates de coupure, politique des commandes multi-vendeurs) sont regroupés dans la cellule 1.

> Exécuter les cellules **dans l'ordre** (Run All). Les fichiers sont écrits dans `/kaggle/working/processed/`.

## 1. Paramètres, imports

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

INPUT_DIR  = os.environ.get("OLIST_DIR", "/kaggle/input/datasets/olistbr/brazilian-ecommerce/")
OUTPUT_DIR = os.environ.get("OUT_DIR", "/kaggle/working/processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- Paramètres de préparation (décisions documentées dans le rapport) ----
PERIOD_START = "2017-01-01"      # mois avec au moins 500 commandes : janvier 2017 à août 2018
PERIOD_END   = "2018-08-31"
TRAIN_END    = "2018-02-28"      # train : jusqu'à cette date incluse
VALID_END    = "2018-04-30"      # valid : jusqu'à cette date incluse ; test : après
MULTI_SELLER_POLICY = "keep_first"   # "keep_first" (vendeur du 1er article) ou "exclude"
SMOOTHING_K        = 10          # lissage bayésien des statistiques de vendeur et de trajet
RARE_CATEGORY_MIN  = 100         # catégories rares regroupées en "other" (comptées sur le train)
EXTREME_DAYS       = 60          # au-delà : livraison marquée "extrême" (conservée, signalée)
BRAZIL_LAT, BRAZIL_LNG = (-34, 6), (-74, -34)
DAY = 86400

print("Entrées :", INPUT_DIR)
print("Sorties :", OUTPUT_DIR)

## 2. Chargement des 9 tables et conversion des types

In [ ]:
def read(name):
    return pd.read_csv(os.path.join(INPUT_DIR, name))

customers   = read("olist_customers_dataset.csv")
geolocation = read("olist_geolocation_dataset.csv")
order_items = read("olist_order_items_dataset.csv")
payments    = read("olist_order_payments_dataset.csv")
reviews     = read("olist_order_reviews_dataset.csv")
orders      = read("olist_orders_dataset.csv")
products    = read("olist_products_dataset.csv")
sellers     = read("olist_sellers_dataset.csv")
category    = read("product_category_name_translation.csv")

for c in ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
          "order_delivered_customer_date", "order_estimated_delivery_date"]:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")
order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"], errors="coerce")
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"], errors="coerce")

# Les codes postaux sont des identifiants : texte sur 5 caractères
for d, col in [(customers, "customer_zip_code_prefix"), (sellers, "seller_zip_code_prefix"),
               (geolocation, "geolocation_zip_code_prefix")]:
    d[col] = d[col].astype(str).str.zfill(5)

print({n: len(d) for n, d in [("orders", orders), ("customers", customers), ("order_items", order_items),
                              ("products", products), ("sellers", sellers), ("reviews", reviews),
                              ("payments", payments), ("geolocation", geolocation)]})

## 3. Sélection des commandes (étape 1)
Seules les commandes **livrées, avec une date de livraison cohérente**, dans la **période retenue**, sont conservées. Le tableau d'entonnoir documente chaque filtre.

In [ ]:
funnel = []
def log_step(label, d):
    funnel.append({"étape": label, "commandes": len(d)})

sel = orders[["order_id", "customer_id", "order_status", "order_purchase_timestamp",
              "order_delivered_customer_date", "order_estimated_delivery_date"]].copy()
log_step("Toutes les commandes", sel)

sel = sel[sel["order_status"] == "delivered"];                                   log_step("Statut « delivered »", sel)
sel = sel[sel["order_delivered_customer_date"].notna()];                        log_step("Avec date de livraison", sel)
sel = sel[sel["order_delivered_customer_date"] >= sel["order_purchase_timestamp"]]; log_step("Livraison après l'achat", sel)

# Historique complet (avant filtre de période) : sert aux statistiques de vendeur et de trajet
hist_orders = sel.copy()

p0, p1 = pd.Timestamp(PERIOD_START), pd.Timestamp(PERIOD_END) + pd.Timedelta(days=1)
sel = sel[(sel["order_purchase_timestamp"] >= p0) & (sel["order_purchase_timestamp"] < p1)]
log_step(f"Période {PERIOD_START} à {PERIOD_END}", sel)
print("Sélection terminée :", len(sel), "commandes")

## 4. Nettoyage de la géolocalisation (étape 3)
Filtre des points hors du Brésil, puis **médiane** des coordonnées par préfixe de code postal (une ligne par préfixe). Un centroïde par état sert de solution de repli.

In [ ]:
geo = geolocation.copy()
in_brazil = geo["geolocation_lat"].between(*BRAZIL_LAT) & geo["geolocation_lng"].between(*BRAZIL_LNG)
print("Lignes dupliquées :", int(geo.duplicated().sum()), "| points hors du Brésil :", int((~in_brazil).sum()))
geo = geo[in_brazil]

geo_zip = (geo.groupby("geolocation_zip_code_prefix")
              .agg(lat=("geolocation_lat", "median"), lng=("geolocation_lng", "median"))
              .reset_index().rename(columns={"geolocation_zip_code_prefix": "zip"}))
geo_state = (geo.groupby("geolocation_state")
                .agg(state_lat=("geolocation_lat", "median"), state_lng=("geolocation_lng", "median"))
                .reset_index().rename(columns={"geolocation_state": "state"}))

print("Préfixes de code postal :", len(geo_zip), "| états :", len(geo_state))

## 5. Nettoyage des produits (étape 2)
Poids nuls traités comme manquants, imputation par la **médiane de la catégorie** (puis médiane globale), catégorie « unknown », traduction des deux catégories sans traduction, correction du nom des colonnes (`lenght`).

In [ ]:
prod = products.rename(columns={"product_name_lenght": "product_name_length",
                                "product_description_lenght": "product_description_length"}).copy()

dims = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
for c in dims:
    prod[c] = prod[c].replace(0, np.nan)          # poids ou dimension nuls = manquants
prod["volume_cm3"] = prod["product_length_cm"] * prod["product_height_cm"] * prod["product_width_cm"]

cat_map = dict(zip(category["product_category_name"], category["product_category_name_english"]))
cat_map.update({"pc_gamer": "pc_gamer",
                "portateis_cozinha_e_preparadores_de_alimentos": "portable_kitchen_food_preparers"})
prod["categorie"] = prod["product_category_name"].map(cat_map).fillna("unknown")

for c in ["product_weight_g", "volume_cm3"]:
    prod[c + "_was_missing"] = prod[c].isna().astype(int)
    by_cat = prod.groupby("categorie")[c].transform("median")
    prod[c] = prod[c].fillna(by_cat).fillna(prod[c].median())

print("Produits :", len(prod), "| poids imputés :", int(prod["product_weight_g_was_missing"].sum()),
      "| volumes imputés :", int(prod["volume_cm3_was_missing"].sum()))

## 6. Agrégation au niveau commande
Une commande peut avoir plusieurs articles, vendeurs et paiements : on agrège **avant** de joindre, pour ne pas dupliquer les lignes. Le **premier article** (`order_item_id` le plus petit) donne le vendeur principal et la catégorie.

In [ ]:
items = (order_items
         .merge(prod[["product_id", "product_weight_g", "volume_cm3", "categorie",
                      "product_weight_g_was_missing"]], on="product_id", how="left")
         .merge(sellers[["seller_id", "seller_zip_code_prefix", "seller_state"]], on="seller_id", how="left")
         .sort_values(["order_id", "order_item_id"]))

agg = items.groupby("order_id").agg(
    nb_items=("order_item_id", "count"),
    nb_sellers=("seller_id", "nunique"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    total_weight_g=("product_weight_g", "sum"),
    total_volume_cm3=("volume_cm3", "sum"),
    weight_imputed=("product_weight_g_was_missing", "max"),
    shipping_limit=("shipping_limit_date", "min"),
).reset_index()

first = (items.groupby("order_id").first()
              [["seller_id", "seller_zip_code_prefix", "seller_state", "categorie"]].reset_index())

pay = (payments.sort_values("payment_value", ascending=False).drop_duplicates("order_id")
               [["order_id", "payment_type"]]
       .merge(payments.groupby("order_id")["payment_installments"].max().reset_index(), on="order_id"))

# Un seul avis par commande : le plus récent (utilisé uniquement pour le modèle 3)
rev = (reviews.sort_values("review_answer_timestamp").drop_duplicates("order_id", keep="last")
              [["order_id", "review_score"]])

cust = customers[["customer_id", "customer_unique_id", "customer_zip_code_prefix", "customer_state"]]
print("Commandes agrégées :", len(agg), "| commandes multi-vendeurs :", int((agg["nb_sellers"] > 1).sum()))

## 7. Assemblage et distance vendeur-client
Jointures (`how="left"` pour ne rien perdre), politique des commandes multi-vendeurs, **distance de Haversine** entre le code postal du vendeur et celui du client. Si un code postal est absent de la géolocalisation, on utilise le centroïde de l'état (colonne `distance_imputed` = 1).

In [ ]:
df = sel.merge(agg, on="order_id", how="inner");  log_step("Avec au moins un article", df)
df = (df.merge(first, on="order_id", how="left")
        .merge(pay, on="order_id", how="left")
        .merge(cust, on="customer_id", how="left"))
df["payment_type"] = df["payment_type"].fillna("unknown")
df["payment_installments"] = df["payment_installments"].fillna(1)
df["customer_state"] = df["customer_state"].fillna("unknown")
df["seller_state"] = df["seller_state"].fillna("unknown")

# Politique multi-vendeurs
df["multi_seller"] = (df["nb_sellers"] > 1).astype(int)
if MULTI_SELLER_POLICY == "exclude":
    df = df[df["multi_seller"] == 0];  log_step("Commandes à un seul vendeur", df)

# Coordonnées
df = df.merge(geo_zip.rename(columns={"zip": "customer_zip_code_prefix", "lat": "customer_lat", "lng": "customer_lng"}),
              on="customer_zip_code_prefix", how="left")
df = df.merge(geo_zip.rename(columns={"zip": "seller_zip_code_prefix", "lat": "seller_lat", "lng": "seller_lng"}),
              on="seller_zip_code_prefix", how="left")
df = df.merge(geo_state.rename(columns={"state": "customer_state", "state_lat": "c_state_lat", "state_lng": "c_state_lng"}),
              on="customer_state", how="left")
df = df.merge(geo_state.rename(columns={"state": "seller_state", "state_lat": "s_state_lat", "state_lng": "s_state_lng"}),
              on="seller_state", how="left")

df["distance_imputed"] = df[["customer_lat", "seller_lat"]].isna().any(axis=1).astype(int)
df["customer_lat"] = df["customer_lat"].fillna(df["c_state_lat"]); df["customer_lng"] = df["customer_lng"].fillna(df["c_state_lng"])
df["seller_lat"]   = df["seller_lat"].fillna(df["s_state_lat"]);   df["seller_lng"]   = df["seller_lng"].fillna(df["s_state_lng"])

def haversine_km(lat1, lng1, lat2, lng2):
    lat1, lng1, lat2, lng2 = map(np.radians, [lat1, lng1, lat2, lng2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lng2 - lng1) / 2) ** 2
    return 2 * 6371.0088 * np.arcsin(np.sqrt(a))

df["distance_km"] = haversine_km(df["seller_lat"], df["seller_lng"], df["customer_lat"], df["customer_lng"])
df = df[df["distance_km"].notna()];  log_step("Avec distance calculable", df)
df = df.reset_index(drop=True)

print("Distances imputées par l'état :", int(df["distance_imputed"].sum()),
      f"({df['distance_imputed'].mean()*100:.2f} %)")
print(df["distance_km"].describe().round(1))

## 8. Variables connues à l'achat
Calendrier, charge de la plateforme (nombre de commandes des 7 jours précédents, **toutes** commandes confondues) et délai laissé au vendeur.

In [ ]:
ts = df["order_purchase_timestamp"]
df["purchase_month"] = ts.dt.month
df["purchase_dow"]   = ts.dt.dayofweek
df["purchase_hour"]  = ts.dt.hour
df["is_weekend"]     = (df["purchase_dow"] >= 5).astype(int)
df["is_black_friday_week"] = ((ts >= "2017-11-20") & (ts < "2017-11-28")).astype(int)

# Charge de la plateforme : commandes passées dans les 7 jours précédant l'achat (information passée uniquement)
all_ts = np.sort(orders["order_purchase_timestamp"].dropna().values.astype("datetime64[ns]"))
t = ts.values.astype("datetime64[ns]")
df["orders_last_7d"] = (np.searchsorted(all_ts, t, side="left")
                        - np.searchsorted(all_ts, t - np.timedelta64(7, "D"), side="left"))

df["limit_days"]   = (df["shipping_limit"] - ts).dt.total_seconds() / DAY     # délai laissé au vendeur pour expédier
df["limit_days"]   = df["limit_days"].fillna(df["limit_days"].median())
df["same_state"]   = (df["seller_state"] == df["customer_state"]).astype(int)
df["freight_ratio"] = df["total_freight"] / df["total_price"].clip(lower=1)

for c in ["total_price", "total_freight", "total_weight_g", "total_volume_cm3", "distance_km"]:
    df["log_" + c] = np.log1p(df[c])

print(df[["purchase_month", "purchase_dow", "purchase_hour", "orders_last_7d", "limit_days", "freight_ratio"]].describe().round(2))

## 9. Variables cibles et découpage temporel
- `delivery_days` = livraison réelle − achat (jours) ; `is_late` = livraison après la date estimée ; `promised_days` = date estimée − achat.
- Découpage **par date d'achat** : train jusqu'à `TRAIN_END`, valid jusqu'à `VALID_END`, test ensuite.
- Les livraisons de plus de 60 jours sont **conservées** et signalées (`is_extreme_delivery`).

In [ ]:
df["delivery_days"]  = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.total_seconds() / DAY
df["promised_days"]  = (df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]).dt.total_seconds() / DAY
df["is_late"]        = (df["order_delivered_customer_date"] > df["order_estimated_delivery_date"]).astype(int)
df["log_delivery_days"]    = np.log1p(df["delivery_days"])
df["is_extreme_delivery"]  = (df["delivery_days"] > EXTREME_DAYS).astype(int)

ts = df["order_purchase_timestamp"]
df["split"] = np.select(
    [ts < pd.Timestamp(TRAIN_END) + pd.Timedelta(days=1), ts < pd.Timestamp(VALID_END) + pd.Timedelta(days=1)],
    ["train", "valid"], default="test")

summary = (df.groupby("split")
             .agg(commandes=("order_id", "count"), debut=("order_purchase_timestamp", "min"),
                  fin=("order_purchase_timestamp", "max"), taux_retard=("is_late", "mean"),
                  duree_mediane=("delivery_days", "median"))
             .reindex(["train", "valid", "test"]))
display(summary)
print("Livraisons extrêmes (> %d j) : %d" % (EXTREME_DAYS, df["is_extreme_delivery"].sum()))

## 10. Historique du vendeur et du trajet, sans fuite de données
Pour chaque commande achetée à la date *t*, on ne compte que les livraisons **terminées avant *t*** (`livraison < t`). Les moyennes sont **lissées** (lissage bayésien, `SMOOTHING_K`) vers la moyenne globale du **train**, pour éviter des valeurs instables sur les vendeurs récents.

In [ ]:
# Historique : toutes les commandes livrées (y compris avant la période retenue)
hist = (hist_orders.merge(first[["order_id", "seller_id", "seller_state"]], on="order_id", how="inner")
                   .merge(cust[["customer_id", "customer_state"]], on="customer_id", how="left"))
hist["delivery_days"] = (hist["order_delivered_customer_date"] - hist["order_purchase_timestamp"]).dt.total_seconds() / DAY
hist["is_late"] = (hist["order_delivered_customer_date"] > hist["order_estimated_delivery_date"]).astype(int)
hist["seller_route"] = hist["seller_state"].astype(str) + "|" + hist["customer_state"].astype(str)
df["seller_route"] = df["seller_state"].astype(str) + "|" + df["customer_state"].astype(str)

train_mask = df["split"] == "train"
prior_days = df.loc[train_mask, "delivery_days"].mean()
prior_late = df.loc[train_mask, "is_late"].mean()
print(f"Priors (train) : durée moyenne {prior_days:.2f} j | taux de retard {prior_late*100:.2f} %")

def past_stats(target, hist, key_col, prefix, prior_days, prior_late, k):
    """Statistiques sur les livraisons DÉJÀ TERMINÉES avant la date d'achat de chaque commande."""
    target = target.reset_index(drop=True)
    t = target["order_purchase_timestamp"].values.astype("datetime64[ns]")
    n = np.zeros(len(target)); sum_d = np.zeros(len(target)); sum_l = np.zeros(len(target))
    h = hist.sort_values("order_delivered_customer_date")
    groups = {key: g for key, g in h.groupby(key_col)}
    for key, idx in target.groupby(key_col).indices.items():
        g = groups.get(key)
        if g is None:
            continue
        done = g["order_delivered_customer_date"].values.astype("datetime64[ns]")
        cum_d = np.cumsum(g["delivery_days"].values)
        cum_l = np.cumsum(g["is_late"].values)
        pos = np.searchsorted(done, t[idx], side="left")          # livraisons terminées strictement avant t
        n[idx] = pos
        sum_d[idx] = np.where(pos > 0, cum_d[np.maximum(pos - 1, 0)], 0.0)
        sum_l[idx] = np.where(pos > 0, cum_l[np.maximum(pos - 1, 0)], 0.0)
    return pd.DataFrame({f"{prefix}_n_past": n,
                         f"{prefix}_mean_days": (sum_d + k * prior_days) / (n + k),
                         f"{prefix}_late_rate": (sum_l + k * prior_late) / (n + k)})

df = df.reset_index(drop=True)
df = pd.concat([df,
                past_stats(df, hist, "seller_id",    "seller", prior_days, prior_late, SMOOTHING_K),
                past_stats(df, hist, "seller_route", "route",  prior_days, prior_late, SMOOTHING_K)], axis=1)

print(df[["seller_n_past", "seller_mean_days", "seller_late_rate", "route_n_past", "route_mean_days", "route_late_rate"]].describe().round(3))
print("Commandes dont le vendeur n'a aucune livraison passée :", int((df["seller_n_past"] == 0).sum()))

## 11. Catégories rares, baselines et contrôle de fuite
- Catégories rares regroupées en `other` (seuil compté sur le **train**).
- **Baselines** pour l'évaluation du modèle 1 : médiane globale et médiane par état du client, calculées sur le **train** uniquement. La promesse d'Olist (`promised_days`) est la troisième baseline.
- Vérification automatique : le vendeur n'utilise jamais une livraison postérieure à l'achat.

In [ ]:
counts = df.loc[train_mask, "categorie"].value_counts()
keep = counts[counts >= RARE_CATEGORY_MIN].index
df["categorie_grouped"] = np.where(df["categorie"].isin(keep), df["categorie"], "other")
print("Catégories conservées :", len(keep), "| regroupées en « other » :", df["categorie"].nunique() - len(keep))

state_median = df.loc[train_mask].groupby("customer_state")["delivery_days"].median()
global_median = df.loc[train_mask, "delivery_days"].median()
df["baseline_state_median_days"]  = df["customer_state"].map(state_median).fillna(global_median)
df["baseline_global_median_days"] = global_median

# Contrôle de fuite : recompter à la main l'historique d'un échantillon de commandes
rng = np.random.default_rng(0)
sample = df.sample(min(200, len(df)), random_state=0)
errors = 0
for _, r in sample.iterrows():
    past = hist[(hist["seller_id"] == r["seller_id"]) & (hist["order_delivered_customer_date"] < r["order_purchase_timestamp"])]
    if len(past) != r["seller_n_past"]:
        errors += 1
print("Contrôle anti-fuite (200 commandes) : erreurs =", errors)
assert errors == 0, "L'historique du vendeur ne correspond pas aux livraisons antérieures à l'achat"

## 12. Jeu de données 1 : durée de livraison (régression)
Variables explicatives **connues à l'achat**, cible `delivery_days`, et baselines. `promised_days` est fournie comme variable **optionnelle** : à comparer avec et sans en modélisation (voir `FEATURES_PROMISE`).

In [ ]:
ID_COLS = ["order_id", "customer_unique_id", "order_purchase_timestamp", "split"]

FEATURES_NUMERIC = [
    "nb_items", "multi_seller", "total_price", "total_freight", "total_weight_g", "total_volume_cm3",
    "weight_imputed", "freight_ratio", "limit_days", "distance_km", "distance_imputed", "same_state",
    "customer_lat", "customer_lng", "seller_lat", "seller_lng", "payment_installments",
    "purchase_month", "purchase_dow", "purchase_hour", "is_weekend", "is_black_friday_week", "orders_last_7d",
    "seller_n_past", "seller_mean_days", "seller_late_rate", "route_n_past", "route_mean_days", "route_late_rate",
    "log_total_price", "log_total_freight", "log_total_weight_g", "log_total_volume_cm3", "log_distance_km",
]
FEATURES_CATEGORICAL = ["customer_state", "seller_state", "categorie_grouped", "payment_type"]
FEATURES_CORE    = FEATURES_NUMERIC + FEATURES_CATEGORICAL     # sans la promesse d'Olist
FEATURES_PROMISE = ["promised_days"]                            # optionnelle (connue à l'achat)

TARGETS_M1   = ["delivery_days", "log_delivery_days", "is_extreme_delivery"]
BASELINES_M1 = ["baseline_state_median_days", "baseline_global_median_days"]

m1 = df[ID_COLS + FEATURES_CORE + FEATURES_PROMISE + TARGETS_M1 + BASELINES_M1].copy()
print("Modèle 1 :", m1.shape, "| manquants dans les variables :", int(m1[FEATURES_CORE + FEATURES_PROMISE].isna().sum().sum()))
display(m1.head(3))

## 13. Jeu de données 2 : risque de retard (classification)
Mêmes variables que le modèle 1 (pipeline partagé), cible `is_late`. La durée réelle **n'est pas** incluse : elle servirait à retrouver la cible. Pour la mini-expérience « régression puis seuil », on rejoint les prédictions du modèle 1 par `order_id`.

In [ ]:
TARGETS_M2 = ["is_late"]
m2 = df[ID_COLS + FEATURES_CORE + FEATURES_PROMISE + TARGETS_M2].copy()

print("Modèle 2 :", m2.shape)
rates = m2.groupby("split")["is_late"].agg(["count", "sum", "mean"]).reindex(["train", "valid", "test"])
rates.columns = ["commandes", "retards", "taux_retard"]
display(rates)

## 14. Jeu de données 3 : segmentation des clients (clustering)
Une ligne par **client réel** (`customer_unique_id`), calculée sur les commandes livrées de la période. Variables : récence, valeur, satisfaction, expérience de livraison, région.
- La **fréquence** est presque toujours 1 (97 % de clients à achat unique) : elle est fournie comme indicateur `is_repeat_customer`, pas comme variable principale.
- Variables asymétriques : transformation `log1p`. Colonnes `*_rs` : mise à l'échelle **robuste** (médiane et écart interquartile), prêtes pour K-Means.
- Les notes manquantes sont imputées par la médiane, avec l'indicateur `review_missing`.

In [ ]:
c3 = df.merge(rev, on="order_id", how="left")
c3["days_late"] = ((c3["order_delivered_customer_date"] - c3["order_estimated_delivery_date"]).dt.total_seconds() / DAY).clip(lower=0)
c3["gap_days"]  = c3["promised_days"] - c3["delivery_days"]          # > 0 : livré en avance
c3 = c3.sort_values("order_purchase_timestamp")

ref_date = c3["order_purchase_timestamp"].max().normalize() + pd.Timedelta(days=1)
m3 = (c3.groupby("customer_unique_id")
        .agg(n_orders=("order_id", "nunique"),
             first_purchase=("order_purchase_timestamp", "min"),
             last_purchase=("order_purchase_timestamp", "max"),
             monetary_total=("total_price", "sum"),
             freight_total=("total_freight", "sum"),
             avg_order_value=("total_price", "mean"),
             avg_items=("nb_items", "mean"),
             avg_delivery_days=("delivery_days", "mean"),
             avg_gap_days=("gap_days", "mean"),
             late_share=("is_late", "mean"),
             max_days_late=("days_late", "max"),
             avg_review_score=("review_score", "mean"),
             n_reviews=("review_score", "count"),
             customer_state=("customer_state", "last"),
             last_category=("categorie_grouped", "last"))
        .reset_index())

m3["recency_days"] = (ref_date - m3["last_purchase"]).dt.total_seconds() / DAY
m3["is_repeat_customer"] = (m3["n_orders"] > 1).astype(int)
m3["review_missing"] = (m3["n_reviews"] == 0).astype(int)
m3["avg_review_score"] = m3["avg_review_score"].fillna(m3["avg_review_score"].median())

REGIONS = {"Norte": ["AC", "AP", "AM", "PA", "RO", "RR", "TO"],
           "Nordeste": ["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"],
           "Centro-Oeste": ["DF", "GO", "MT", "MS"],
           "Sudeste": ["ES", "MG", "RJ", "SP"],
           "Sul": ["PR", "RS", "SC"]}
state_to_region = {s: r for r, states in REGIONS.items() for s in states}
m3["region"] = m3["customer_state"].map(state_to_region).fillna("unknown")

m3["log_monetary"]     = np.log1p(m3["monetary_total"])
m3["log_recency"]      = np.log1p(m3["recency_days"])
m3["log_max_days_late"] = np.log1p(m3["max_days_late"])

CLUSTER_FEATURES = ["recency_days", "log_monetary", "avg_review_score",
                    "avg_delivery_days", "log_max_days_late", "avg_gap_days"]
def robust_scale(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = (q3 - q1) if (q3 - q1) > 0 else (s.std() if s.std() > 0 else 1.0)
    return (s - s.median()) / iqr
for c in CLUSTER_FEATURES:
    m3[c + "_rs"] = robust_scale(m3[c])
CLUSTER_FEATURES_RS = [c + "_rs" for c in CLUSTER_FEATURES]

print("Modèle 3 :", m3.shape, "| clients à achat répété :", f"{m3['is_repeat_customer'].mean()*100:.2f} %",
      "| sans avis :", f"{m3['review_missing'].mean()*100:.2f} %")
print("Date de référence de la récence :", ref_date.date())
display(m3[CLUSTER_FEATURES].describe().round(2))

## 15. Contrôles de qualité avant sauvegarde

In [ ]:
FORBIDDEN = {"order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date",
             "review_score", "gap_days", "seller_days", "carrier_days", "delivery_days", "log_delivery_days",
             "is_late", "is_extreme_delivery"}
features_all = FEATURES_CORE + FEATURES_PROMISE
assert not (set(features_all) & FORBIDDEN), "Fuite : une variable interdite figure parmi les features"
assert m1[features_all].isna().sum().sum() == 0, "Valeurs manquantes dans les features"
assert m1["order_id"].is_unique and m2["order_id"].is_unique, "order_id dupliqué"
assert set(m1["split"].unique()) == {"train", "valid", "test"}, "Un des trois blocs est vide"
mx = lambda s: m1.loc[m1["split"] == s, "order_purchase_timestamp"]
assert mx("train").max() < mx("valid").min() and mx("valid").max() < mx("test").min(), "Les périodes se chevauchent"
assert m1[["delivery_days"]].min().iloc[0] >= 0, "Durée négative"
assert m3["customer_unique_id"].is_unique, "Client dupliqué dans le modèle 3"
assert m3[CLUSTER_FEATURES_RS].isna().sum().sum() == 0, "Valeurs manquantes dans les variables de clustering"

report = pd.DataFrame(funnel)
report["perte"] = report["commandes"].diff().fillna(0).astype(int)
print("Entonnoir de sélection :")
display(report)
print("Tous les contrôles sont validés.")

## 16. Visualisation de contrôle : les blocs train / valid / test sont-ils comparables ?

In [ ]:
order = ["train", "valid", "test"]
fig, ax = plt.subplots(1, 3, figsize=(18, 4))

(m2.groupby("split")["is_late"].mean().reindex(order) * 100).plot.bar(ax=ax[0], color="tab:blue")
ax[0].set_title("Taux de retard (%) par bloc"); ax[0].tick_params(axis="x", rotation=0)

for s in order:
    ax[1].hist(m1.loc[m1["split"] == s, "delivery_days"].clip(upper=60), bins=40, alpha=0.5, density=True, label=s)
ax[1].set_title("Durée de livraison par bloc (jours, tronquée à 60)"); ax[1].legend()

m3["recency_days"].clip(upper=700).plot.hist(bins=40, ax=ax[2], color="tab:green")
ax[2].set_title("Récence des clients (jours)")
plt.tight_layout(); plt.show()

print("Médiane de la durée par bloc :")
print(m1.groupby("split")["delivery_days"].median().reindex(order).round(2))

## 17. Dictionnaire des données et sauvegarde des 3 jeux de données

In [ ]:
DESC = {
 "order_id": "Identifiant de la commande", "customer_unique_id": "Identifiant du client réel",
 "order_purchase_timestamp": "Date et heure d'achat", "split": "Bloc temporel : train, valid ou test",
 "nb_items": "Nombre d'articles de la commande", "multi_seller": "1 si la commande a plusieurs vendeurs",
 "total_price": "Prix total des articles (BRL)", "total_freight": "Frais de port totaux (BRL)",
 "total_weight_g": "Poids total (g), imputé si manquant", "total_volume_cm3": "Volume total (cm3), imputé si manquant",
 "weight_imputed": "1 si le poids d'au moins un produit a été imputé", "freight_ratio": "Frais de port / prix",
 "limit_days": "Délai laissé au vendeur pour expédier (jours)", "distance_km": "Distance vendeur-client (Haversine, km)",
 "distance_imputed": "1 si la distance repose sur le centroïde de l'état", "same_state": "1 si vendeur et client sont dans le même état",
 "customer_lat": "Latitude du client", "customer_lng": "Longitude du client", "seller_lat": "Latitude du vendeur", "seller_lng": "Longitude du vendeur",
 "payment_installments": "Nombre de mensualités (paiement principal)", "purchase_month": "Mois d'achat (1 à 12)",
 "purchase_dow": "Jour de la semaine (0 = lundi)", "purchase_hour": "Heure d'achat", "is_weekend": "1 si achat le week-end",
 "is_black_friday_week": "1 si achat entre le 20 et le 27 novembre 2017", "orders_last_7d": "Commandes de la plateforme dans les 7 jours précédents",
 "seller_n_past": "Livraisons terminées du vendeur avant l'achat", "seller_mean_days": "Durée moyenne passée du vendeur (lissée)",
 "seller_late_rate": "Taux de retard passé du vendeur (lissé)", "route_n_past": "Livraisons terminées du trajet (état vendeur vers état client) avant l'achat",
 "route_mean_days": "Durée moyenne passée du trajet (lissée)", "route_late_rate": "Taux de retard passé du trajet (lissé)",
 "log_total_price": "log1p du prix total", "log_total_freight": "log1p des frais de port", "log_total_weight_g": "log1p du poids",
 "log_total_volume_cm3": "log1p du volume", "log_distance_km": "log1p de la distance",
 "customer_state": "État du client", "seller_state": "État du vendeur (1er article)",
 "categorie_grouped": "Catégorie du 1er article (rares regroupées en other)", "payment_type": "Mode de paiement principal",
 "promised_days": "Durée promise par Olist (date estimée - achat), connue à l'achat. OPTIONNELLE",
 "delivery_days": "CIBLE modèle 1 : durée réelle de livraison (jours)", "log_delivery_days": "log1p de la durée réelle",
 "is_extreme_delivery": "1 si la durée dépasse 60 jours", "baseline_state_median_days": "Baseline : médiane du train par état du client",
 "baseline_global_median_days": "Baseline : médiane globale du train", "is_late": "CIBLE modèle 2 : 1 si livré après la date estimée",
 "n_orders": "Nombre de commandes livrées", "first_purchase": "Première date d'achat", "last_purchase": "Dernière date d'achat",
 "monetary_total": "Montant total dépensé (BRL)", "freight_total": "Frais de port totaux (BRL)", "avg_order_value": "Panier moyen (BRL)",
 "avg_items": "Nombre moyen d'articles par commande", "avg_delivery_days": "Durée de livraison moyenne vécue (jours)",
 "avg_gap_days": "Écart moyen promesse - réel (> 0 : en avance)", "late_share": "Part des commandes livrées en retard",
 "max_days_late": "Retard maximal vécu (jours, 0 si jamais en retard)", "avg_review_score": "Note moyenne (imputée par la médiane si absente)",
 "n_reviews": "Nombre d'avis laissés", "last_category": "Catégorie de la dernière commande", "recency_days": "Jours depuis la dernière commande",
 "is_repeat_customer": "1 si plus d'une commande", "review_missing": "1 si le client n'a laissé aucun avis", "region": "Région du Brésil",
 "log_monetary": "log1p du montant dépensé", "log_recency": "log1p de la récence", "log_max_days_late": "log1p du retard maximal",
}
for c in CLUSTER_FEATURES:
    DESC[c + "_rs"] = f"{c} mise à l'échelle robuste (médiane et écart interquartile)"

datasets = {"model1_delivery_duration": m1, "model2_late_risk": m2, "model3_customer_segmentation": m3}
rows = []
for name, d in datasets.items():
    for c in d.columns:
        rows.append({"dataset": name, "variable": c, "type": str(d[c].dtype), "description": DESC.get(c, "")})
dictionary = pd.DataFrame(rows)
missing_desc = dictionary.loc[dictionary["description"] == "", "variable"].unique().tolist()
assert not missing_desc, f"Variables sans description : {missing_desc}"

for name, d in datasets.items():
    d.to_csv(os.path.join(OUTPUT_DIR, name + ".csv"), index=False)
dictionary.to_csv(os.path.join(OUTPUT_DIR, "data_dictionary.csv"), index=False)
report.to_csv(os.path.join(OUTPUT_DIR, "selection_funnel.csv"), index=False)

metadata = {
    "parameters": {"period": [PERIOD_START, PERIOD_END], "train_end": TRAIN_END, "valid_end": VALID_END,
                   "multi_seller_policy": MULTI_SELLER_POLICY, "smoothing_k": SMOOTHING_K,
                   "rare_category_min": RARE_CATEGORY_MIN, "extreme_days": EXTREME_DAYS},
    "model1": {"file": "model1_delivery_duration.csv", "target": "delivery_days", "target_log": "log_delivery_days",
               "features_core": FEATURES_CORE, "features_optional_promise": FEATURES_PROMISE,
               "categorical": FEATURES_CATEGORICAL, "baselines": BASELINES_M1 + ["promised_days"]},
    "model2": {"file": "model2_late_risk.csv", "target": "is_late", "features_core": FEATURES_CORE,
               "features_optional_promise": FEATURES_PROMISE, "categorical": FEATURES_CATEGORICAL},
    "model3": {"file": "model3_customer_segmentation.csv", "cluster_features": CLUSTER_FEATURES,
               "cluster_features_scaled": CLUSTER_FEATURES_RS, "reference_date": str(ref_date.date())},
    "rows": {k: int(len(v)) for k, v in datasets.items()},
    "forbidden_in_features": sorted(FORBIDDEN),
}
with open(os.path.join(OUTPUT_DIR, "metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Fichiers écrits dans", OUTPUT_DIR)
for fn in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  {fn:40s} {os.path.getsize(os.path.join(OUTPUT_DIR, fn))/1024:>10,.0f} Ko")

## Synthèse de la phase Data Preparation

| Jeu de données | Lignes | Cible | Variables |
|---|---|---|---|
| `model1_delivery_duration.csv` | une par commande livrée | `delivery_days` | 34 numériques, 4 catégorielles, `promised_days` optionnelle |
| `model2_late_risk.csv` | une par commande livrée | `is_late` | idem (sans la durée réelle) |
| `model3_customer_segmentation.csv` | une par client réel | aucune | 6 variables de clustering et leurs versions `*_rs` |

**À retenir pour la modélisation (phase 4)**
- Évaluer sur `valid`, puis une seule fois sur `test`. Ajuster les hyperparamètres avec une validation croisée **temporelle** sur `train`.
- Comparer le modèle 1 aux baselines : `promised_days`, `baseline_global_median_days`, `baseline_state_median_days`.
- Entraîner les modèles 1 et 2 **avec et sans** `promised_days` (variable connue à l'achat mais qui reflète les règles internes d'Olist).
- Le taux de retard varie fortement d'un bloc à l'autre : évaluer le modèle 2 avec PR-AUC, rappel et précision, pas avec l'exactitude.
- Encodage des catégories (target encoding, one-hot) : à faire **dans** la validation croisée, pas ici.
- Segmentation : utiliser les colonnes `*_rs` avec K-Means, puis profiler chaque cluster.